In [1]:
######################################################################
## 03_ANALYSIS: Calculating CpG o/e Ratio and Average Methylation - PYTHON
######################################################################

# This notebook computes CpG observed/expected (CpG o/e) for gene bodies (CDS) and promoters, overlaps each with WGBS methylation data. These steps are done in python before moving to R.

In [1]:
######################################################################
## BLOCK 0. Project setup (relative paths; no private paths)
######################################################################

from pathlib import Path
import sys, platform

# Assumes you run this notebook from the repository/project root.
PROJECT_DIR = Path('.').resolve()
DATA_DIR    = PROJECT_DIR / 'data'
OUT_DIR     = PROJECT_DIR / 'outputs'
FIG_DIR     = OUT_DIR / 'figures'
DERIVED_DIR = OUT_DIR / 'derived'

for d in [OUT_DIR, FIG_DIR, DERIVED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Expected:
#   data/wgbs/   : per-sample CpG tables (gzipped) used below
#   data/    : genome FASTA + CDS fasta (+ optional promoter fasta)
# If your files live elsewhere, create symlinks into these folders.


In [2]:
######################################################################
## BLOCK 1. Setup & Imports
######################################################################

import pandas as pd
import re
import numpy as np
from pathlib import Path
from Bio import SeqIO
import pyranges as pr

In [3]:
######################################################################
## BLOCK 2. Define Inputs
######################################################################

from pathlib import Path
import pandas as pd

# Reference / annotation inputs (not tracked in git; place or symlink into data/ref/)
cds_fasta    = DATA_DIR / "cds_from_genomic.fna"
genome_fasta = DATA_DIR / "mytilus_californianus_genome.fasta"
gff_file     = DATA_DIR / "GCF_021869535.1_genomic.gff"

# WGBS per-sample CpG tables (4× filtered, SNP-removed), e.g. *.cov_4x.tab.gz
# Option A (recommended): provide a metadata TSV with explicit sample names:
#   data/metadata/wgbs_cov_files.tsv with columns: sample, path
meta_tsv = DATA_DIR / "metadata" / "wgbs_cov_files.tsv"

if meta_tsv.exists():
    meta = pd.read_csv(meta_tsv, sep="\t")
    file_paths   = [str(PROJECT_DIR / p) for p in meta["path"].tolist()]
    sample_names = meta["sample"].tolist()
else:
    wgbs_dir = DATA_DIR / "wgbs"
    file_paths = sorted([str(p) for p in wgbs_dir.glob("*.cov_4x.tab.gz")])
    # Fallback: derive sample names from filenames
    sample_names = [Path(p).name.replace(".cov_4x.tab.gz","") for p in file_paths]

# Basic existence checks (fail early with a helpful message)
for p in [cds_fasta, genome_fasta, gff_file]:
    if not Path(p).exists():
        raise FileNotFoundError(f"Missing required input: {p}\n"
                                f"Place/symlink it under {DATA_DIR/'ref'} or update paths in BLOCK 2.")

if len(file_paths) == 0:
    raise FileNotFoundError(f"No WGBS files found. Put *.cov_4x.tab.gz under {DATA_DIR/'wgbs'} "
                            "or provide data/metadata/wgbs_cov_files.tsv.")


In [4]:
######################################################################
## BLOCK 3. Load & Filter WGBS Methylation
######################################################################

dfs = []
for fp, name in zip(file_paths, sample_names):
    df = pd.read_csv(fp, sep="\t", compression="gzip", header=None,
                     names=["chrom","start","end","meth_percent","methylated","unmethylated"])
    df["sample"] = name
    df["methylated"]   = pd.to_numeric(df["methylated"],   errors="coerce")
    df["unmethylated"] = pd.to_numeric(df["unmethylated"], errors="coerce")
    df["total_reads"]  = df["methylated"] + df["unmethylated"]
    df = df[df["total_reads"] >= 4]
    df["meth_percent"] = 100.0 * df["methylated"] / df["total_reads"]
    dfs.append(df)

all_methyl = pd.concat(dfs, ignore_index=True)
all_methyl = all_methyl.rename(columns={"chrom":"Chromosome","start":"Start","end":"End"})
print("WGBS loaded:", all_methyl.shape)
print(all_methyl["meth_percent"].describe())

WGBS loaded: (91506363, 8)
count    9.150636e+07
mean     1.085309e+01
std      2.667312e+01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.000000e+02
Name: meth_percent, dtype: float64


In [5]:
######################################################################
## BLOCK 4. Compute CpG o/e for CDS
######################################################################

cds_oe = []
for rec in SeqIO.parse(cds_fasta, "fasta"):
    seq = str(rec.seq).upper()
    cg, c, g, L = seq.count("CG"), seq.count("C"), seq.count("G"), len(seq)
    oe = (cg/(c*g))*((L**2)/(L-1)) if (c>0 and g>0 and L>1) else float("nan")
    cds_oe.append({"cds_id": rec.id, "cpg_oe": oe})
cpg_oe_cds_df = pd.DataFrame(cds_oe)
print("CDS CpG o/e:", cpg_oe_cds_df.shape)
cpg_oe_cds_df.head()

CDS CpG o/e: (49739, 2)


,cds_id,cpg_oe
0,lcl|NW_026262581.1_cds_XP_052072905.1_1,0.770346
1,lcl|NW_026262581.1_cds_XP_052089389.1_2,0.121006
2,lcl|NW_026262581.1_cds_XP_052089418.1_3,0.119629
3,lcl|NW_026262581.1_cds_XP_052089483.1_4,0.056816
4,lcl|NW_026262581.1_cds_XP_052089280.1_5,0.413299


In [6]:
######################################################################
## BLOCK 5. Parse CDS Metadata
######################################################################

import re
from Bio import SeqIO
import pandas as pd

cds_meta = []

for rec in SeqIO.parse(cds_fasta, "fasta"):
    h  = rec.description
    cm = re.search(r"lcl\|([^_]+_[^_]+\.\d+)", h)               # Chromosome ID
    sm = re.search(r"location=(\d+)", h)                        # Start
    em = re.search(r"\.\.(\d+)", h)                             # End
    gid = re.search(r"\[gene=([^\]]+)\]", h)                    # Gene ID

    if not (cm and sm and em and gid):
        continue

    # Clean the gene_id to remove "LOC" and retain only digits
    raw_gene_id = gid.group(1)
    numeric_gene_id = re.search(r'(\d+)$', raw_gene_id)
    if not numeric_gene_id:
        continue

    cds_meta.append({
        "cds_id": rec.id,
        "Chromosome": cm.group(1),
        "Start": int(sm.group(1)),
        "End":   int(em.group(1)),
        "gene_id": numeric_gene_id.group(1)
    })

cds_df = pd.DataFrame(cds_meta)
print("Parsed CDS metadata:", cds_df.shape)
cds_df.head()

Parsed CDS metadata: (2200, 5)


,cds_id,Chromosome,Start,End,gene_id
0,lcl|NW_026262581.1_cds_XP_052072905.1_1,NW_026262581.1,330767,331333,127711003
1,lcl|NW_026262581.1_cds_XP_052070843.1_8,NW_026262581.1,563787,564527,127706699
2,lcl|NW_026262581.1_cds_XP_052093523.1_23,NW_026262581.1,1540537,1541541,127729688
3,lcl|NW_026262581.1_cds_XP_052096334.1_47,NW_026262581.1,1880499,1882598,127731580
4,lcl|NW_026262581.1_cds_XP_052085922.1_105,NW_026262581.1,3812208,3814292,127723123


In [7]:
######################################################################
## BLOCK 6. Overlap CDS & Compute Avg Methylation → Save
######################################################################

# Give each CpG site a 1-bp range for PyRanges
meth_df = all_methyl.copy()
meth_df["End"] = meth_df["Start"]

# Build PyRanges objects
meth_ranges = pr.PyRanges(meth_df[["Chromosome", "Start", "End", "meth_percent"]])
cds_ranges  = pr.PyRanges(cds_df[["Chromosome", "Start", "End", "cds_id", "gene_id"]])

# Join CpGs to CDS intervals
cds_over = meth_ranges.join(cds_ranges)

# Average methylation per CDS
cds_meth_avg = (
    cds_over.df
    .groupby(["cds_id", "gene_id"], as_index=False)["meth_percent"]
    .mean()
    .rename(columns={"meth_percent": "avg_methylation"})
)

# Merge with CpG o/e table (if you want to include that)
merged_cds = pd.merge(cpg_oe_cds_df, cds_meth_avg, on="cds_id", how="inner")

# Write out final CDS methylation file
out1 = DATA_DIR / "cpg_methylation_merged.tsv"
merged_cds.to_csv(out1, sep="\t", index=False)
print("Merged CDS:", merged_cds.shape)
merged_cds.head()

Merged CDS: (2113, 4)


,cds_id,cpg_oe,gene_id,avg_methylation
0,lcl|NW_026262581.1_cds_XP_052072905.1_1,0.770346,127711003,0.000
1,lcl|NW_026262581.1_cds_XP_052070843.1_8,0.827366,127706699,0.000
2,lcl|NW_026262581.1_cds_XP_052093523.1_23,1.058453,127729688,0.000
3,lcl|NW_026262581.1_cds_XP_052085922.1_105,0.740205,127723123,0.425
4,lcl|NW_026262581.1_cds_XP_052057634.1_131,0.809870,127698280,0.000


In [8]:
######################################################################
## BLOCK 7. Define Promoters & Clamp to Scaffold Lengths
######################################################################

gff = pd.read_csv(gff_file, sep="\t", comment="#", header=None,
                  names=["seqid","src","type","start","end","score","strand","phase","attr"])
genes = gff[gff["type"].isin(["gene","mRNA"])].copy()
genes["gene_id"] = genes["attr"].str.extract(r"GeneID:([0-9]+)")

def get_promoter(row):
    if row["strand"] == "+":
        ps, pe = row["start"] - 1000, row["start"] - 1
    else:
        ps, pe = row["end"] + 1, row["end"] + 1000
    return pd.Series({
        "promoter_start": max(1, ps),
        "promoter_end": pe
    })

prom_coords = genes.apply(get_promoter, axis=1)
prom_df = pd.concat([genes[["seqid","strand","gene_id"]], prom_coords], axis=1)
prom_df = prom_df.rename(columns={"seqid":"Chromosome","gene_id":"promoter_id"})

# clamp to scaffold lengths
scf_len = {r.id: len(r.seq) for r in SeqIO.parse(genome_fasta, "fasta")}
prom_df["Start"] = prom_df.apply(lambda r: min(r.promoter_start, scf_len[r.Chromosome]), axis=1)
prom_df["End"]   = prom_df.apply(lambda r: min(r.promoter_end,   scf_len[r.Chromosome]), axis=1)

# write BED for external extraction
prom_df[["Chromosome","Start","End","promoter_id"]].to_csv(
    "promoters.bed", sep="\t", index=False, header=False
)
print("Promoters prepared:", prom_df.shape)
prom_df.head()

Promoters prepared: (90393, 7)


,Chromosome,strand,promoter_id,promoter_start,promoter_end,Start,End
1,NW_026262581.1,+,127715832,1,396,1,396
4,NW_026262581.1,+,127716489,1516,2515,1516,2515
7,NW_026262581.1,+,127716907,3634,4633,3634,4633
10,NW_026262581.1,+,127714569,5748,6747,5748,6747
13,NW_026262581.1,+,127714643,7866,8865,7866,8865


In [9]:
######################################################################
## BLOCK 8. Extract Promoter Sequences
######################################################################

from Bio import SeqIO
from Bio.SeqRecord import SeqRecord

# Load the whole genome into a dict (once)
genome_dict = SeqIO.to_dict(SeqIO.parse(genome_fasta, "fasta"))

# Extract each promoter by slicing the SeqRecord
promoter_records = []
for _, row in prom_df.iterrows():
    chrom = row.Chromosome
    # SeqIO is 0-based, end exclusive
    seq = genome_dict[chrom].seq[row.Start - 1 : row.End]
    promoter_records.append(
        SeqRecord(seq, id=row.promoter_id, description="")
    )

# Write out a FASTA of all promoters
SeqIO.write(promoter_records, "promoter_seqs.fasta", "fasta")
print(f"Extracted {len(promoter_records)} promoter sequences.")

Extracted 90393 promoter sequences.


In [10]:
######################################################################
## BLOCK 9. Compute CpG o/e for Promoters
######################################################################

prom_oe = []
for rec in SeqIO.parse("promoter_seqs.fasta", "fasta"):
    pid = rec.id
    s = str(rec.seq).upper()
    cg = s.count("CG")
    c  = s.count("C")
    g  = s.count("G")
    L  = len(s)
    oe = (cg/(c*g)) * ((L**2)/(L-1)) if (c>0 and g>0 and L>1) else float("nan")
    prom_oe.append({"promoter_id": pid, "cpg_oe": oe})

prom_oe_df = pd.DataFrame(prom_oe)
print("Promoter CpG o/e:", prom_oe_df.shape)
prom_oe_df.head()

Promoter CpG o/e: (90393, 2)


,promoter_id,cpg_oe
0,127715832,0.487598
1,127716489,0.473685
2,127716907,0.470789
3,127714569,0.473361
4,127714643,0.473361


In [11]:
######################################################################
## BLOCK 10. Overlap Promoter Methylation & Merge → Save
######################################################################

import pyranges as pr

# 10a) Build PyRanges for WGBS methylation
meth_ranges = pr.PyRanges(
    all_methyl[["Chromosome", "Start", "End", "meth_percent"]]
)

# 10b) Build PyRanges for promoters: select only Chromosome/Start/End/promoter_id
# and rename 'promoter_id' → 'Name' so PyRanges sees the ID column correctly.
prom_ranges = pr.PyRanges(
    prom_df[["Chromosome", "Start", "End", "promoter_id"]]
        .rename(columns={"promoter_id": "Name"})
)

# 10c) Join and compute mean methylation per promoter
overlap_prom = meth_ranges.join(prom_ranges)
prom_meth_avg = (
    overlap_prom.df
    .groupby("Name", as_index=False)["meth_percent"]
    .mean()
    .rename(columns={"Name": "promoter_id", "meth_percent": "avg_methylation"})
)

# 10d) Merge with promoter CpG o/e and save
merged_prom = pd.merge(
    prom_oe_df, prom_meth_avg,
    on="promoter_id", how="inner"
)
merged_prom.to_csv(
    "promoter_cpg_methylation_merged.tsv",
    sep="\t", index=False
)

print("Merged promoters:", merged_prom.shape)
merged_prom.head()

Merged promoters: (84510, 3)


,promoter_id,cpg_oe,avg_methylation
0,127716907,0.470789,0.000000
1,127714569,0.473361,0.000000
2,127714940,0.475962,5.000000
3,127715878,0.477957,0.000000
4,127716668,0.470789,0.022114


In [12]:
######################################################################
## BLOCK 11. Filter & Save CDS Table
######################################################################

# Remove CDS with zero average methylation
filtered_cds = merged_cds[merged_cds["avg_methylation"] > 0].copy()
print("Filtered CDS:", filtered_cds.shape)

# Save to file — keeps gene_id, cds_id, and avg_methylation
out_cds  = DATA_DIR / "cpg_methylation_filtered.tsv"
filtered_cds.to_csv(out_cds, sep="\t", index=False)

Filtered CDS: (1830, 4)


In [13]:
######################################################################
## BLOCK 12. Filter & Save Promoter Table
######################################################################

from pathlib import Path

# Step 1: Filter to only promoter_ids that match gene_ids in CDS
cds_ids = set(cds_df["gene_id"])
filtered_prom = merged_prom[merged_prom["promoter_id"].isin(cds_ids)].copy()

# Step 2: Drop zero methylation (optional)
filtered_prom = filtered_prom[filtered_prom["avg_methylation"] > 0]

# Step 3: Drop any remaining duplicates by promoter_id (1 per gene)
filtered_prom = filtered_prom.sort_values("avg_methylation", ascending=False)
filtered_prom = filtered_prom.drop_duplicates("promoter_id")

# Step 4: Save to file
out_prom = DATA_DIR / "promoter_cpg_methylation_filtered.tsv"
filtered_prom.to_csv(out_prom, sep="\t", index=False)

print("Filtered & deduplicated promoters:", filtered_prom.shape)

Filtered & deduplicated promoters: (1143, 3)


In [18]:
######################################################################
## BLOCK 13. CpG o/e vs EXPRESSION inputs
##   150 bp proximal-promoter o/e + full-span gene-body o/e.
##   Writes two tables consumed by notebook 06 for Fig. S3.
######################################################################
import pandas as pd
from Bio import SeqIO

# Reuse genome if already loaded; otherwise load it.
try:
    genome_dict
except NameError:
    genome_dict = SeqIO.to_dict(SeqIO.parse(genome_fasta, "fasta"))
scf_len = {k: len(v.seq) for k, v in genome_dict.items()}

def cpg_oe(seq):
    seq = seq.upper()
    cg, c, g, L = seq.count("CG"), seq.count("C"), seq.count("G"), len(seq)
    return (((cg / (c * g)) * ((L ** 2) / (L - 1))) if (c > 0 and g > 0 and L > 1)
            else float("nan")), cg

# One row per gene, numeric GeneID, on scaffolds present in the FASTA
g = gff[gff["type"] == "gene"].copy()
g["gene_id"] = g["attr"].str.extract(r"GeneID:([0-9]+)")
g = g.dropna(subset=["gene_id"])
g = g[g["seqid"].isin(genome_dict.keys())]

# --- (a) 150 bp proximal promoter o/e (strand-aware, clamped to scaffold) ---
prom_rows = []
for _, r in g.iterrows():
    if r["strand"] == "+":
        ps, pe = r["start"] - 150, r["start"] - 1
    else:
        ps, pe = r["end"] + 1, r["end"] + 150
    ps, pe = max(1, ps), min(pe, scf_len[r["seqid"]])
    if pe < ps:
        continue
    seq = str(genome_dict[r["seqid"]].seq[ps - 1:pe])
    oe, cg = cpg_oe(seq)
    prom_rows.append({"gene_id": r["gene_id"], "cpg_oe_150": oe,
                      "n_CpG": cg, "L": len(seq)})
prom150 = pd.DataFrame(prom_rows).drop_duplicates("gene_id")
prom150.to_csv(DATA_DIR / "promoter_cpgoe_150.tsv", sep="\t", index=False)

# --- (b) full-span gene-body o/e (TSS–TTS) ---
gb_rows = []
for _, r in g.iterrows():
    seq = str(genome_dict[r["seqid"]].seq[int(r["start"]) - 1:int(r["end"])])
    oe, _ = cpg_oe(seq)
    gb_rows.append({"gene_id": r["gene_id"], "cpg_oe_gb": oe})
genebody = pd.DataFrame(gb_rows).drop_duplicates("gene_id")
genebody.to_csv(DATA_DIR / "genebody_cpgoe_full.tsv", sep="\t", index=False)

# --- power check ---
nd = int(prom150["cpg_oe_150"].notna().sum())
nz = int((prom150["cpg_oe_150"] > 0).sum())
print(f"Promoters (150 bp): total={len(prom150)}, defined o/e={nd}, non-zero={nz}")
print(f"Gene bodies (full span): total={len(genebody)}")
print("Wrote promoter_cpgoe_150.tsv and genebody_cpgoe_full.tsv")

Promoters (150 bp): total=40654, defined o/e=40648, non-zero=36429
Gene bodies (full span): total=40654
Wrote promoter_cpgoe_150.tsv and genebody_cpgoe_full.tsv


In [15]:
######################################################################
## Reproducibility
######################################################################

import sys
import platform
import importlib

print("Date:", __import__("datetime").date.today())
print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())
print()

pkgs = [
    "numpy",
    "pandas",
    "Bio",
    "pyranges"
]

print("Key package versions:")
for p in pkgs:
    try:
        m = importlib.import_module(p)
        print(f"  {p:12s} {m.__version__}")
    except Exception:
        print(f"  {p:12s} not installed")

Date: 2026-07-30
Python: 3.11.5 (main, Sep 11 2023, 13:54:46) [GCC 11.2.0]
Platform: Linux-5.15.0-185-generic-x86_64-with-glibc2.35

Key package versions:
  numpy        1.26.3
  pandas       2.1.4
  Bio          1.83
  pyranges     0.1.4
